# 🎓 Educational Deep Dive: Medallion Architecture

Welcome to the Medallion Pipeline bootcamp! 
The **Medallion Architecture** is a data design pattern used to logically organize data in a Lakehouse. It consists of three layers:
1. **🥉 Bronze (Raw):** The landing zone. Data is ingested as-is, typically in append-only mode. It's immutable and acts as our historical archive.
2. **🥈 Silver (Cleansed):** The filtering zone. We deduplicate, enforce schemas, and often flatten nested structures (like exploding JSON arrays) to form Star Schemas.
3. **🥇 Gold (Business-Ready):** The serving zone. Here we apply business rules, aggregations, and data governance (like PII Masking) so the data is ready for BI dashboards.

Let's build this step by step!

# Databricks Data Engineering Bootcamp
## Case Study: Global Electronics Production Pipeline

**Task:** Welcome to the Global Electronics engineering team! As our Lead Data Engineer, you are responsible for building our production-grade data platform on Databricks from scratch. You will deploy a comprehensive production-grade Medallion Architecture Pipeline (Bronze -> Silver -> Gold) within Databricks, transforming raw ingested structures into a highly optimized Star Schema layout tailored for downstream BI utilities.

---

## Chapter 0: The Architecture Sandbox (Theoretical Foundations)

**Context:** Before implementing the Medallion Pipeline, we analyze how different structural models behave under distributed query conditions and map key constraints.

### Exercise 0.1: Reading the Sandbox Catalog Tables
Initialize the Spark DataFrames by reading directly from the managed enterprise Unity Catalog schema (ai_lab.default).

In [0]:
from pyspark.sql import functions as F

In [0]:
# TODO: Fill in the correct Spark execution function and table names to read the source tables
customers_sandbox = spark.read.table("ai_lab.default.customers")
products_sandbox  = spark.read.table("ai_lab.default.products")
orders_sandbox    = spark.read.table("ai_lab.default.orders")

In [0]:
# Add a column with the source file path for every DataFrame
customers_sandbox = customers_sandbox.withColumn("source_file_path", F.col("_metadata.file_path"))
products_sandbox  = products_sandbox.withColumn("source_file_path", F.col("_metadata.file_path"))
orders_sandbox    = orders_sandbox.withColumn("source_file_path", F.col("_metadata.file_path"))

In [0]:
# Check out the initial state of the DataFrames
display(customers_sandbox.limit(3))
display(products_sandbox.limit(3))
display(orders_sandbox.limit(3))

In [0]:
# Inspect the schemas for data characterization
customers_sandbox.printSchema()
products_sandbox.printSchema()
orders_sandbox.printSchema()

### Exercise 0.2: Database Key Constraints Identification
Before moving assets into staging layers, analyze and profile the underlying unique keys.

In [0]:
# --- STEP 1: Unique Key Validation for Customers & Products ---

# TODO: Group by business keys to validate identity constraints across dimensions
dup_customers = customers_sandbox.groupBy("customer_id").count().filter("count > 1").count()
dup_products  = products_sandbox.groupBy("product_id").count().filter("count > 1").count()

print(f"-> Found {dup_customers} duplicate customer IDs.")
print(f"-> Found {dup_products} duplicate product IDs.")

### 👨‍🏫 Instructor Note: The Power of Explode
In modern data lakes, we often receive nested JSON (e.g., an Order contains a list of Items). To build a relational **Star Schema**, we must flatten this. The `F.explode()` function takes an array column and creates a new row for each element in the array. This transforms our document-style data into a classic relational Fact Table!

In [0]:
# --- STEP 2: Cleansing & Composite Key Construction for Orders ---

# TODO: Filter out records with missing structural identifiers
orders_base = orders_sandbox.filter(F.col("order_id").isNotNull())

# TODO: Remove redundant transaction records to enforce snapshot isolation rules
orders_deduped = orders_base.dropDuplicates(["order_id", "customer_id", "order_date"])

# TODO: Flatten nested arrays to align dataframe to line-item grain boundaries
orders_exploded = orders_deduped.withColumn("item", F.explode("items")) \
                                .withColumn("product_id", F.col("item.product_id")) \
                                .withColumn("quantity", F.col("item.quantity"))

# TODO: Synthesize the final composite primary key for orders using unique transactional attributes
silver_orders_final = orders_exploded.withColumn(
    "order_item_pk", 
    F.concat_ws("-", F.col("order_id"), F.col("product_id"))
).select(
    "order_item_pk", 
    "order_id", 
    "customer_id", 
    "order_date", 
    "product_id", 
    "quantity"
)

# TODO: Target validation check to audit schema integrity and grain output
print("\nFinal Silver Orders with Composite Primary Key:")
display(silver_orders_final)

---

## Chapter 1: Landing the Data (Bronze Layer)

**Context:** Raw unstructured/semi-structured files have arrived in cloud object storage. The orders dataset (`orders.json`) contains nested structures.

### Exercise 1.1: Ingestion & Storage Specification
Persist the operational data assets into the Bronze Layer using the Delta lake open format with an Append save mode configuration.

### 👨‍🏫 Instructor Note: Writing to Bronze
Notice how we use `.format("delta")` and `.mode("append")`. The Bronze layer should **never** overwrite data. We just append new batches as they arrive. Delta Lake ensures ACID compliance even if multiple streams are writing simultaneously.

In [0]:
# TODO: Complete the write configuration parameters specifying a delta layout with an append mode and schema evolution
customers_sandbox.write.format("delta") \
    .mode("append") \
    .saveAsTable("bronze_customers")

# TODO: Complete the write configuration parameters specifying a delta layout with an append mode and schema evolution
orders_sandbox.write.format("delta") \
    .mode("append") \
    .saveAsTable("bronze_orders")

# TODO: Complete the write configuration parameters specifying a delta layout with an append mode and schema evolution
products_sandbox.write.format("delta") \
    .mode("append") \
    .saveAsTable("bronze_products")

In [0]:
# Verify catalog registration
print("Registered Bronze Layer Tables:")
display(spark.sql("SHOW TABLES LIKE 'bronze_*'"))

## Chapter 2: Data Quality, Cleansing & Dimensional Modeling (Silver Layer)

### Architectural Context: Silver Star Schema Modeling
Based on corporate data platform standards, multi-dimensional Star Schema modeling must take place within the Silver Layer. 
Here, we sanitize incoming structural records and organize them into standardized analytical models (Facts & Dimensions):

![Global Electronics Star Schema](star_schema.png)

1. Fact Table (silver_fact_sales): Captures numerical and quantifiable business events.
   - The Grain: 1 row per singular Order Line Item. If a transaction contains 3 different items, it generates 3 distinct rows.
2. Dimension Tables (silver_customers, silver_dim_products): Hold descriptive, textual attributes providing business context.

---
### Exercise 2.1: Data Profiling & Deduplication
Quantify duplicate record occurrences and execute data quality remediation rules to drop Null structures from the Bronze Orders staging table.


In [0]:
# TODO: Profile duplicate frequencies by grouping the business transactional identifier fields
bronze_orders = spark.table("bronze_orders")

duplicate_count = bronze_orders.groupBy("order_id").count().filter("count > 1").count()
print(f"Profiling Metric - Duplicate Order IDs: {duplicate_count}")

# Execute isolation and sanitization rules to remediate null values
silver_orders_clean = bronze_orders.filter(F.col("order_id").isNotNull()).dropDuplicates(["order_id"])

### Exercise 2.2: Building the Dimensions & Fact Tables (Silver Star Schema Deployment)
**Operational Challenge (Handling Semi-Structured Nested JSON):** Our raw order source records store product transactions inside a nested array format named `items`. In order to align with our analytical **Fact Table Grain (1 row per Order Line Item)**, we must flatten this structure.

We use the PySpark `F.explode()` function to unpack the nested array elements into individual top-level database rows. Once exploded, we can successfully resolve the `product_id` key and join our Fact table with the `silver_dim_products` dimension.

### 👨‍🏫 Instructor Note: Writing to Bronze
Notice how we use `.format("delta")` and `.mode("append")`. The Bronze layer should **never** overwrite data. We just append new batches as they arrive. Delta Lake ensures ACID compliance even if multiple streams are writing simultaneously.

In [0]:
# 1. Customer Dimension Table
silver_customers = spark.table("bronze_customers").dropDuplicates(["customer_id"])
silver_customers.write.format("delta") \
    .option("mergeSchema", "true") \
    .mode("overwrite") \
    .saveAsTable("silver_dim_customers")

# 2. Product Dimension Table
silver_products = spark.table("bronze_products").dropDuplicates(["product_id"])
silver_products.write.format("delta") \
    .option("mergeSchema", "true") \
    .mode("overwrite") \
    .saveAsTable("silver_dim_products")

# 3. Central Fact Table Execution (Exploding nested items to a granular Order Line-Item level)
# TODO: Ensure all incoming columns and the new composite PK are preserved
silver_fact_sales = spark.table("bronze_orders").filter(F.col("order_id").isNotNull()) \
    .dropDuplicates(["order_id", "customer_id", "order_date"]) \
    .withColumn("item", F.explode("items")) \
    .withColumn("product_id", F.col("item.product_id")) \
    .withColumn("quantity", F.col("item.quantity")) \
    .withColumn("order_item_pk", F.concat_ws("-", F.col("order_id"), F.col("product_id"))) \
    .join(silver_products, "product_id", "inner") \
    .select(
        "order_item_pk", 
        "order_id",
        "order_date",
        "customer_id",
        "product_id",
        "quantity"
    )

silver_fact_sales.write.format("delta") \
    .option("mergeSchema", "true") \
    .mode("overwrite") \
    .saveAsTable("silver_fact_sales")

print("Silver Layer Status: Dimensional Star Schema Deployed Successfully.")

_If you want to check how Historicity (SCD Type 2) and Data Vault modeling work in practice before moving forward, pause here and jump into notebooks 02_deep_dive_scd2_historization and 03_deep_dive_data_vault_modeling.
Otherwise, if you just want to stick to the core track and build the final Star Schema, keep scrolling and dive straight into Chapter 3._

---

## Chapter 3: Analytics Modeling & Serving (Gold Layer)

### Architectural Context: Gold Layer Analytical Specialization
The Gold Layer does NOT serve as an entity modeling playground. It hosts highly curated, 
use-case specific tables or virtualized view abstractions customized to feed reporting tools and analytical dashboards.

![Data Governance Delivery Flow](data_governance.png)

### Data Governance & PII Anonymization Masking
To satisfy enterprise regulatory compliance frameworks, you must apply conditional column-level masking logic. 
Anonymize customer identity features by obfuscating strings, exposing only the first 3 characters followed by a mask marker.

### Conceptual Check (Tables vs Views):
Core Interview Prompt: Why do we prioritize a virtual VIEW over a physical TABLE for final BI reporting end-points?
Answer: Views do not consume physical storage, evaluate logic on-demand, and decouple application layers from data architecture modifications. If schema alterations impact silver foundations, we adjust view query parameters without breaking legacy analytical applications.

---
### Exercise 3.1: Serving View Deployment with PII Masking
Deploy the final reporting presentation boundary view layer by introducing a substring column-mask expression.

### 👨‍🏫 Instructor Note: Data Governance (PII Masking)
In the Gold layer, we expose data to business users. However, we must comply with privacy laws (GDPR, CCPA). Here, we dynamically mask the `customer_name` using `SUBSTR` and `CONCAT` so that analysts can see the trends without exposing Personal Identifiable Information (PII).

In [0]:
# TODO: Construct the string masking concatenation expression using substring formatting rules
spark.sql("""
CREATE OR REPLACE VIEW v_serving_sales_performance AS
SELECT 
    f.order_id AS Order_Number,
    f.order_date AS Order_Date,
    CONCAT(SUBSTR(c.customer_name, 1, 3), '***') AS Masked_Customer_Name,
    c.city AS Delivery_City,
    p.product_name AS Product_Name,
    p.category AS Product_Category,
    f.quantity AS Units_Sold,
    (f.quantity * p.price) AS Total_Revenue
FROM silver_fact_sales f
INNER JOIN silver_customers c ON f.customer_id = c.customer_id
INNER JOIN silver_dim_products p ON f.product_id = p.product_id
""")

print("Serving Status: Business View Endpoint Online (PII Masking Rule Enforcement Active).")
display(spark.table("v_serving_sales_performance"))